# MNV3-S crop-context comparison — seed 100, 12 epochs

New paired recipe: MobileNetV3-Small Vanilla vs frozen CSMR-v1, using **2.7×** face context and 224×224 RGB. The frozen Stage-0 seed42 Train/Val manifest and LCC set/cache remain unchanged. This is a practical recipe comparison: crop context and training budget both differ from the old 1.55×/6-epoch run, so it is not a pure crop or architecture ablation.

This notebook reuses the completed seed-100 implementation for the MNV3-S architecture/head, ImageNet initialization, normalization, augmentation, optimizer, PAD score, metrics, Val-only selection/thresholding, LCC evaluation, frozen CSMR-v1 loss, diagnostics, and serialization. Crop records retain their old bbox/status/provenance; a separate adapter ignores the old model-specific crop factor and applies 2.7× with the same square crop, reflection boundary handling, RGB conversion, resize/letterbox, and normalization. Stage-0 files are read-only.

Historical summary paths and fixed opened Test-2000 manifest/cache paths are optional exact file paths in the config cell. Leave them blank to omit those comparisons. No dataset directory is searched recursively.


In [ ]:
# DATA ROOTS / KAGGLE CONFIG — change paths only for the attached datasets.
STAGE0_RESOURCE_DIR = (
    "/kaggle/input/datasets/maithanhphuong/"
    "face-auth-stage0-micro-v1"
)
STAGE1_RESULT_DIR = (
    "/kaggle/input/datasets/maithanhphuong/"
    "smartface-pad-micro-stage1-v1/"
    "stage1_shared_micro_e1"
)
CELEBA_ROOT = (
    "/kaggle/input/datasets/attentionlayer241/"
    "celeba-spoof-for-face-antispoofing/"
    "CelebA_Spoof_/CelebA_Spoof"
)
LCC_FASD_ROOT = (
    "/kaggle/input/datasets/aleksandrpikul222/"
    "lcc-fasd/"
    "LCC_FASD"
)
IMAGENET_WEIGHTS_PATH = ""
# Existing buffalo_s model root containing models/buffalo_s/det_500m.onnx.
# Attach the exact Stage-0 detector model if it is not present here.
SCRFD_MODEL_ROOT = "/root/.insightface"
# Optional exact JSON summaries for historical reference rows; no filesystem search.
OLD_MNV3_SUMMARY_PATH = ""
MINIFASNET_SUMMARY_PATH = ""
# Optional exact prior opened Test-2000 files; leave both blank to skip Test.
FIXED_TEST_MANIFEST_PATH = ""
FIXED_TEST_BBOX_CACHE_PATH = ""
# Ordered key fingerprint of the Test-2000 already opened in the completed seed100 notebook.
PREVIOUS_OPENED_TEST_ORDERED_SHA256 = "df3416273c3e7d8bf0caa3b010454ee73236211e034ec12bfb481f056fe237b8"
OUTPUT_ROOT = "/kaggle/working/mnv3s_crop27_csmr_seed100"

SEED = 100  # Training RNG only; Stage 0 still uses micro_manifest_v1_seed42.npz.
MNV3_CROP_SCALE = 2.7
BATCH_SIZE = 128  # Only 64 is allowed for an OOM rerun.
NUM_WORKERS = 4
AMP = True
MAX_EPOCHS = 12
EARLY_STOP_PATIENCE = 3
EARLIEST_EARLY_STOP_EPOCH = 6
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
GRAD_CLIP = 5.0
CSMR_WARMUP_EPOCHS = 1
CSMR_LAMBDA = 0.20
CSMR_BAND_CENTERS = [0.15, 0.45, 0.75]
CSMR_BAND_SIGMA = 0.10
CSMR_GAIN_RANGE = [0.65, 0.90]
CSMR_PRESERVE_DC = True
SAVE_INTERVENTION_GALLERY = True
SELECTION_CHUNK_SIZE = 32
TEST_SUBSET_N = 2000
TEST_SUBSET_SEED = 4242


In [ ]:
import csv
import hashlib
import json
import math
import os
import random
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import MobileNet_V3_Small_Weights

CELEBA_MEAN = np.asarray([0.5931, 0.4690, 0.4229], dtype=np.float32)
CELEBA_STD = np.asarray([0.2471, 0.2214, 0.2157], dtype=np.float32)
INPUT_SIZE = 224
E1_ARCHITECTURE = "MobileNetV3-Small-GAP-576-256-BN-HSwish-Dropout02-128-3-v1"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def find_unique_file(name, root=None, explicit_dir=""):
    if not explicit_dir:
        raise RuntimeError(f"Set the explicit directory containing {name}; broad Kaggle search is disabled")
    base = Path(explicit_dir).expanduser()
    path = base if base.is_file() else base / name
    if path.name != name or not path.is_file():
        raise FileNotFoundError(f"Expected {name} at {path}")
    return path.resolve()


def find_stage0_dir():
    root = Path(STAGE0_RESOURCE_DIR).expanduser()
    if not STAGE0_RESOURCE_DIR or not root.is_dir():
        raise FileNotFoundError(f"Stage 0 directory missing: {root}")
    if (root / "stage0_summary.json").is_file():
        return root.resolve()
    if (root / "stage0" / "stage0_summary.json").is_file():
        return (root / "stage0").resolve()
    raise FileNotFoundError(f"stage0_summary.json missing directly under {root} or {root / 'stage0'}")


def find_celeba_root():
    root = Path(CELEBA_ROOT).expanduser()
    if not (root / "metas/intra_test/train_label.json").is_file():
        raise FileNotFoundError(f"CelebA root is wrong: {root}")
    return root.resolve()


def find_lcc_root(manifest_path):
    root = Path(LCC_FASD_ROOT).expanduser()
    if not root.is_dir():
        raise FileNotFoundError(f"LCC root is wrong: {root}")
    return root.resolve()

CELEBA_BBOX_STATUS_FACTORS = {
    "VALID": 1.55,
    "FALLBACK_DET_FAIL": 1.50,
    "FALLBACK_SUSPICIOUS": 1.50,
}


def validate_bbox_record(record, domain, key):
    if not isinstance(record, dict):
        raise TypeError("BBox record must be a mapping for {}".format(key))
    status = str(record.get("status", ""))
    if domain == "celeba":
        if status == "INVALID":
            raise RuntimeError("Selected micro key has INVALID E1 bbox record: {}".format(key))
        if status not in CELEBA_BBOX_STATUS_FACTORS:
            raise RuntimeError("Unknown/unusable E1 bbox status {} for {}".format(status, key))
        expected_factor = CELEBA_BBOX_STATUS_FACTORS[status]
    elif domain == "lcc":
        if status != "VALID":
            raise RuntimeError("Only Stage 0 LCC VALID records may be scored: {} ({})".format(key, status))
        expected_factor = 1.55
    else:
        raise ValueError("Unknown PAD crop domain: {}".format(domain))
    if "crop_factor" not in record:
        raise KeyError("BBox record is missing crop_factor for {}".format(key))
    try:
        crop_factor = float(record["crop_factor"])
    except (TypeError, ValueError) as exc:
        raise ValueError("Invalid crop_factor for {}: {!r}".format(key, record.get("crop_factor"))) from exc
    if not np.isfinite(crop_factor) or not np.isclose(crop_factor, expected_factor, rtol=0.0, atol=1e-6):
        raise ValueError(
            "BBox status/crop-factor mismatch for {}: status={} factor={} expected={}".format(
                key, status, crop_factor, expected_factor
            )
        )
    bbox = np.asarray(record.get("bbox_xyxy", []), dtype=np.float64)
    if bbox.shape != (4,) or not np.isfinite(bbox).all() or bbox[2] <= bbox[0] or bbox[3] <= bbox[1]:
        raise ValueError("Invalid bbox_xyxy for {}: {!r}".format(key, record.get("bbox_xyxy")))
    return crop_factor


def ordered_path_fingerprint(paths):
    return sha256_text("\n".join(map(str, paths)))


def assert_ordered_keys(actual, expected, name):
    actual = list(map(str, actual))
    expected = list(map(str, expected))
    if actual != expected:
        raise RuntimeError(
            "{} sample keys/order differ from the frozen manifest: actual_n={}, expected_n={}".format(
                name, len(actual), len(expected)
            )
        )


def labels_from_attack_codes(codes):
    codes = np.asarray(codes, dtype=np.int64)
    labels = np.full(codes.shape, 2, dtype=np.int64)
    labels[codes == 0] = 0
    labels[np.isin(codes, np.arange(1, 8))] = 1
    return labels

def load_stage0_resources():
    stage0 = find_stage0_dir()
    manifest_path = stage0 / "micro_manifest_v1_seed42.npz"
    bbox_path = stage0 / "celeba_micro_bbox_cache.json"
    lcc_manifest_path = stage0 / "lcc_external_dev_manifest.csv"
    lcc_cache_path = stage0 / "lcc_scrfd_cache_micro_v1.json"
    fingerprints_path = stage0 / "micro_resource_fingerprints.json"
    summary_path = stage0 / "stage0_summary.json"
    for path in [manifest_path, bbox_path, lcc_manifest_path, lcc_cache_path, fingerprints_path, summary_path]:
        if not path.is_file():
            raise FileNotFoundError("Stage 0 resource missing: {}".format(path))
    fingerprints = json.loads(fingerprints_path.read_text())
    stage0_summary = json.loads(summary_path.read_text())
    if stage0_summary.get("schema") != "pad_micro_stage0_summary_v1":
        raise RuntimeError("Unsupported Stage 0 summary schema")
    for path, key in [
        (manifest_path, "micro_manifest_sha256"),
        (lcc_manifest_path, "lcc_manifest_sha256"),
        (lcc_cache_path, "lcc_cache_sha256"),
        (bbox_path, "celeba_bbox_cache_sha256"),
    ]:
        if sha256_file(path) != fingerprints.get(key):
            raise RuntimeError("Stage 0 SHA256 mismatch for {}".format(key))
    with np.load(manifest_path, allow_pickle=False) as pack:
        train_keys = pack["micro_train_keys"].astype(str)
        val_keys = pack["micro_val_keys"].astype(str)
        train_labels = pack["micro_train_labels"].astype(np.int64)
        val_labels = pack["micro_val_labels"].astype(np.int64)
        train_codes = pack["micro_train_attack_codes"].astype(np.int64)
        val_codes = pack["micro_val_attack_codes"].astype(np.int64)
    if len(train_keys) != 5000 or len(val_keys) != 2000:
        raise RuntimeError("Stage 0 must freeze exactly 5,000 Train and 2,000 Val keys")
    if len(set(train_keys) & set(val_keys)):
        raise RuntimeError("Stage 0 Train/Val keys overlap")
    for name, labels, codes in [
        ("Train", train_labels, train_codes),
        ("Val", val_labels, val_codes),
    ]:
        if labels.shape != codes.shape or set(map(int, labels)) != {0, 1, 2}:
            raise RuntimeError("Stage 0 {} label/code arrays are malformed or miss a class".format(name))
        if not np.array_equal(labels, labels_from_attack_codes(codes)):
            raise RuntimeError("Stage 0 {} labels do not match E1 v5.3 attack-code mapping".format(name))
    bbox_payload = json.loads(bbox_path.read_text())
    bbox_cache = bbox_payload.get("records", {})
    selected_keys = set(map(str, train_keys)) | set(map(str, val_keys))
    if bbox_payload.get("schema") != "celeba_micro_bbox_cache_v1" or set(bbox_cache) != selected_keys:
        raise RuntimeError("Stage 0 CelebA bbox cache keys/schema differ from the frozen micro manifest")
    for key, record in bbox_cache.items():
        validate_bbox_record(record, "celeba", key)
    lcc_df = pd.read_csv(lcc_manifest_path)
    lcc_payload = json.loads(lcc_cache_path.read_text())
    lcc_records = lcc_payload.get("records", {})
    if lcc_payload.get("schema") != "pad_micro_lcc_scrfd_cache_v1":
        raise RuntimeError("Unsupported Stage 0 LCC cache schema")
    if lcc_df["path"].astype(str).duplicated().any():
        raise RuntimeError("Stage 0 LCC manifest contains duplicate paths")
    if not set(lcc_df["label"].astype(int)).issubset({0, 1}):
        raise RuntimeError("Stage 0 LCC labels must be exactly 0=Real or 1=Attack")
    record_by_path = {}
    for record in lcc_records.values():
        path = str(record.get("path", ""))
        record_by_path.setdefault(path, []).append(record)
    for row in lcc_df.itertuples(index=False):
        matches = record_by_path.get(str(row.path), [])
        if len(matches) != 1:
            raise RuntimeError("Stage 0 LCC cache join for {} has {} rows".format(row.path, len(matches)))
        record = matches[0]
        if str(record.get("status", "")) != str(row.status):
            raise RuntimeError("Stage 0 LCC manifest/cache status mismatch for {}".format(row.path))
        if int(record.get("label", -1)) != int(row.label):
            raise RuntimeError("Stage 0 LCC manifest/cache label mismatch for {}".format(row.path))
        if str(record.get("frame_id", "")) != str(row.record_id):
            raise RuntimeError("Stage 0 LCC manifest record_id mismatch for {}".format(row.path))
        if str(row.status) == "VALID":
            validate_bbox_record(record, "lcc", row.path)
    valid_paths = lcc_df.loc[lcc_df["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    valid_labels = lcc_df.loc[lcc_df["status"].astype(str) == "VALID", "label"].astype(int).tolist()
    if not valid_paths or set(valid_labels) != {0, 1}:
        raise RuntimeError("Common LCC VALID set must be non-empty and contain Real plus Attack")
    if len(lcc_df) != int(fingerprints.get("lcc_candidate_n", -1)):
        raise RuntimeError("Stage 0 LCC candidate count mismatch")
    if len(valid_paths) != int(fingerprints.get("lcc_valid_face_n", -1)):
        raise RuntimeError("Stage 0 LCC valid-face count mismatch")
    valid_path_sha = ordered_path_fingerprint(valid_paths)
    if valid_path_sha != fingerprints.get("lcc_valid_path_fingerprint_sha256"):
        raise RuntimeError("Stage 0 ordered LCC VALID path fingerprint mismatch")
    if stage0_summary.get("lcc_valid_path_fingerprint_sha256") != valid_path_sha:
        raise RuntimeError("Stage 0 summary ordered LCC VALID path fingerprint mismatch")
    return {
        "stage0_dir": stage0,
        "train_keys": train_keys,
        "val_keys": val_keys,
        "train_labels": train_labels,
        "val_labels": val_labels,
        "train_codes": train_codes,
        "val_codes": val_codes,
        "bbox_cache": bbox_cache,
        "lcc_df": lcc_df,
        "lcc_cache": lcc_records,
        "lcc_valid_paths": valid_paths,
        "fingerprints": fingerprints,
    }
def assert_raw_image_paths(resources, celeba_root, lcc_root):
    celeba_keys = list(map(str, resources["train_keys"])) + list(map(str, resources["val_keys"]))
    lcc_paths = resources["lcc_df"]["path"].astype(str).tolist()
    missing_celeba = [str(Path(celeba_root) / key) for key in celeba_keys if not (Path(celeba_root) / key).is_file()]
    missing_lcc = [str(Path(lcc_root) / path) for path in lcc_paths if not (Path(lcc_root) / path).is_file()]
    if missing_celeba or missing_lcc:
        raise FileNotFoundError(
            "Raw PAD image preflight failed before training: CelebA missing={}, LCC missing={}".format(
                missing_celeba[:5], missing_lcc[:5]
            )
        )
    print("Verified raw image paths before training: CelebA selected={}, LCC candidates={}".format(
        len(celeba_keys), len(lcc_paths)
    ))


def runtime_crop_bgr(image, bbox_xyxy, factor):
    h, w = image.shape[:2]
    x1, y1, x2, y2 = map(float, bbox_xyxy)
    bw, bh = x2 - x1, y2 - y1
    if bw <= 0 or bh <= 0:
        raise ValueError(f"Invalid xyxy bbox: {bbox_xyxy}")
    extent = max(bw, bh) * float(factor)
    side = int(extent)
    cx, cy = x1 + bw / 2.0, y1 + bh / 2.0
    xs, ys = int(cx - extent / 2.0), int(cy - extent / 2.0)
    xa, ya, xb, yb = max(0, xs), max(0, ys), min(w, xs + side), min(h, ys + side)
    if xb <= xa or yb <= ya:
        raise RuntimeError("Empty crop intersection")
    crop = image[ya:yb, xa:xb]
    return cv2.copyMakeBorder(
        crop, max(0, -ys), max(0, ys + side - h),
        max(0, -xs), max(0, xs + side - w), cv2.BORDER_REFLECT_101
    )

def resize_letterbox_rgb(rgb, size=224):
    h, w = rgb.shape[:2]
    ratio = size / max(h, w)
    nh, nw = max(1, int(h * ratio)), max(1, int(w * ratio))
    interp = cv2.INTER_LANCZOS4 if ratio > 1 else cv2.INTER_AREA
    image = cv2.resize(rgb, (nw, nh), interpolation=interp)
    dh, dw = size - nh, size - nw
    return cv2.copyMakeBorder(image, dh // 2, dh - dh // 2, dw // 2, dw - dw // 2, cv2.BORDER_REFLECT_101)

def apply_e1_mild_augmentation(rgb):
    if random.random() < 0.5:
        rgb = cv2.flip(rgb, 1)
    if random.random() < 0.30:
        alpha = random.uniform(0.90, 1.10)
        beta = random.uniform(-0.10, 0.10) * 255.0
        rgb = np.clip(rgb.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return rgb

def bbox_jitter(record):
    x1, y1, x2, y2 = map(float, record["bbox_xyxy"])
    w, h = x2 - x1, y2 - y1
    if random.random() < 0.20:
        base = max(w, h)
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        scale = random.uniform(0.95, 1.05)
        cx += random.uniform(-0.05, 0.05) * base
        cy += random.uniform(-0.05, 0.05) * base
        w, h = w * scale, h * scale
        return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
    return [x1, y1, x2, y2]

def rgb_tensor(rgb):
    unit = rgb.astype(np.float32) / 255.0
    unit = (unit - CELEBA_MEAN) / CELEBA_STD
    return torch.from_numpy(np.ascontiguousarray(unit.transpose(2, 0, 1), dtype=np.float32))

def optional_crop_index(stage0_dir):
    path = Path(stage0_dir) / "crop_cache" / "index.json"
    return json.loads(path.read_text()) if path.is_file() else {}

def read_rgb_crop(image_path, record, domain, key, crop_index, stage0_dir, augment=False):
    crop_factor = validate_bbox_record(record, domain, key)
    cache_key = "{}::{}".format(domain, key)
    if not augment and cache_key in crop_index:
        path = Path(stage0_dir) / "crop_cache" / crop_index[cache_key]
        bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if bgr is None:
            raise FileNotFoundError("Optional Stage 0 crop cache entry is unreadable: {}".format(path))
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(image_path)
    box = bbox_jitter(record) if augment else record["bbox_xyxy"]
    crop = runtime_crop_bgr(bgr, box, crop_factor)
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    if augment:
        rgb = apply_e1_mild_augmentation(rgb)
    return resize_letterbox_rgb(rgb, INPUT_SIZE)
class CelebAMicroDataset(Dataset):
    def __init__(self, keys, labels, codes, root, bbox_records, stage0_dir, train=False):
        self.keys = list(map(str, keys))
        self.labels = np.asarray(labels, dtype=np.int64)
        self.codes = np.asarray(codes, dtype=np.int64)
        self.root = Path(root)
        self.records = bbox_records
        self.stage0_dir = Path(stage0_dir)
        self.crop_index = optional_crop_index(stage0_dir)
        self.train = bool(train)
        if len(self.keys) != len(self.labels) or len(self.keys) != len(self.codes):
            raise ValueError("CelebA key/label/code lengths differ")
        missing = [k for k in self.keys if k not in self.records]
        if missing:
            raise RuntimeError(f"{len(missing)} CelebA keys lack Stage 0 bbox records")
    def __len__(self):
        return len(self.keys)
    def __getitem__(self, i):
        key = self.keys[i]
        rgb = read_rgb_crop_mnv3_27(
            self.root / key, self.records[key], "celeba", key, augment=self.train
        )
        return rgb_tensor(rgb), int(self.labels[i]), key, int(self.codes[i])

class LCCValidDataset(Dataset):
    def __init__(self, frame, root, records, stage0_dir):
        self.frame = frame.reset_index(drop=True)
        self.root = Path(root)
        self.records = records
        self.stage0_dir = Path(stage0_dir)
        self.crop_index = optional_crop_index(stage0_dir)
        self.rows = []
        by_path = {}
        for rec in records.values():
            by_path.setdefault(str(rec.get("path", "")), []).append(rec)
        for _, row in self.frame.iterrows():
            if str(row["status"]) != "VALID":
                continue
            path = str(row["path"])
            candidates = by_path.get(path, [])
            if len(candidates) != 1:
                raise RuntimeError(f"LCC cache join for {path} has {len(candidates)} records")
            self.rows.append((path, int(row["label"]), candidates[0]))
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        key, label, record = self.rows[i]
        rgb = read_rgb_crop_mnv3_27(
            self.root / key, record, "lcc", key, augment=False
        )
        return rgb_tensor(rgb), int(label), key, record.get("status", "VALID")

class SpatialE1(nn.Module):
    def __init__(self, imagenet_state=None):
        super().__init__()
        base = models.mobilenet_v3_small(weights=None)
        if imagenet_state is not None:
            missing, unexpected = base.load_state_dict(imagenet_state, strict=False)
            allowed_missing = [k for k in missing if k.startswith("classifier.")]
            bad_missing = [k for k in missing if k not in allowed_missing]
            allowed_unexpected = [k for k in unexpected if k.startswith("classifier.")]
            bad_unexpected = [k for k in unexpected if k not in allowed_unexpected]
            if bad_missing or bad_unexpected:
                raise RuntimeError(
                    "ImageNet MobileNetV3-Small state_dict is incomplete/incompatible; "
                    "backbone missing={}, unexpected={}".format(bad_missing[:5], bad_unexpected[:5])
                )
        self.features = base.features
        self.avgpool = base.avgpool
        self.projection = nn.Sequential(
            nn.Linear(576, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.Hardswish(),
            nn.Dropout(0.2),
            nn.Linear(128, 3),
        )
    def forward_features(self, x):
        x = self.features(x)
        x = self.avgpool(x).flatten(1)
        return self.projection(x)
    def forward(self, x):
        feature = self.forward_features(x)
        return self.classifier(feature)

def load_imagenet_weights(local_path=""):
    if local_path:
        state = torch.load(local_path, map_location="cpu", weights_only=False)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        if not isinstance(state, dict):
            raise RuntimeError("Local ImageNet weights must be a state_dict")
        return state
    try:
        return MobileNet_V3_Small_Weights.DEFAULT.get_state_dict(progress=True)
    except Exception as exc:
        raise RuntimeError(
            "ImageNet MobileNetV3-Small weights are unavailable. Enable Kaggle Internet "
            "or set IMAGENET_WEIGHTS_PATH to a compatible torchvision state_dict. "
            "Random initialization is intentionally not used."
        ) from exc

def binary_score(logits):
    logits = torch.as_tensor(logits)
    if logits.ndim != 2 or logits.shape[1] != 3:
        raise ValueError("PAD logits must have shape [N, 3] for Real/Physical/Digital")
    if not torch.isfinite(logits).all():
        raise ValueError("PAD logits contain NaN or Infinity")
    return logits[:, 0] - torch.logsumexp(logits[:, 1:], dim=1)
def safe_auc(y_real, scores):
    from sklearn.metrics import roc_auc_score
    y_real = np.asarray(y_real, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y_real.ndim != 1 or scores.ndim != 1 or len(y_real) != len(scores) or len(y_real) == 0:
        raise ValueError("AUC needs non-empty one-dimensional labels and scores of equal length")
    if not np.isfinite(scores).all():
        raise ValueError("AUC scores contain NaN or Infinity")
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("AUC requires both binary classes (positive class is Real)")
    return float(roc_auc_score(y_real, scores))
def threshold_from_val(y3, scores):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("Threshold calibration needs non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("CelebA labels must be 0=Real, 1=Physical, 2=Digital")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("Threshold calibration requires at least one Real and one Attack sample")
    if not np.isfinite(scores).all():
        raise ValueError("Threshold calibration scores contain NaN or Infinity")
    values = np.unique(scores)
    lower = np.nextafter(values[0], -np.inf)
    upper = np.nextafter(values[-1], np.inf)
    if not np.isfinite(lower):
        lower = values[0]
    if not np.isfinite(upper):
        upper = values[-1]
    midpoints = values[:-1] / 2.0 + values[1:] / 2.0
    candidates = np.concatenate(([lower], midpoints, [upper]))
    best = None
    for threshold in candidates:
        pred_real = scores >= threshold
        real = y_real == 1
        attack = y_real == 0
        bpcer = float(np.mean(~pred_real[real]))
        apcer = float(np.mean(pred_real[attack]))
        acer = 0.5 * (apcer + bpcer)
        key = (acer, abs(float(threshold)), float(threshold))
        if best is None or key < best[0]:
            best = (key, float(threshold))
    if best is None or not np.isfinite(best[1]):
        raise RuntimeError("Threshold search did not produce a finite threshold")
    return best[1]
def metrics_binary(y3, scores, threshold, coverage=None, candidate_n=None):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("PAD metrics need non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("PAD labels must be from the frozen class mapping 0/1/2")
    if not np.isfinite(scores).all() or not np.isfinite(float(threshold)):
        raise ValueError("PAD scores and threshold must be finite")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("PAD metrics require at least one Real and one Attack sample")
    if coverage is not None and (not np.isfinite(float(coverage)) or not 0.0 <= float(coverage) <= 1.0):
        raise ValueError("Detector coverage must be finite and within [0, 1]")
    scored_n = len(y3)
    candidate_n = int(candidate_n if candidate_n is not None else scored_n)
    if candidate_n < scored_n:
        raise ValueError("candidate_n cannot be less than n_scored")
    pred_real = scores >= float(threshold)
    real = y_real == 1
    attack = y_real == 0
    bpcer = float(np.mean(~pred_real[real]))
    apcer = float(np.mean(pred_real[attack]))
    acer = 0.5 * (apcer + bpcer)
    return {
        "n_scored": int(scored_n),
        "candidate_n": candidate_n,
        "detector_coverage": float(coverage if coverage is not None else 1.0),
        "apcer": apcer,
        "bpcer": bpcer,
        "acer": acer,
        "hter": acer,
        "auc": safe_auc(y_real, scores),
        "accuracy": float(np.mean(pred_real == y_real)),
        "locked_logit_threshold": float(threshold),
    }
def metrics_celeba(y3, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    result = metrics_binary(y3, scores, threshold, coverage=1.0, candidate_n=len(y3))
    result["accuracy_3class"] = float(np.mean(np.argmax(logits, axis=1) == np.asarray(y3)))
    return result, scores

def make_prediction_frame(keys, y3, codes, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    pred_real = scores >= threshold
    result = pd.DataFrame({
        "key": list(map(str, keys)),
        "true_class": np.asarray(y3, dtype=np.int64),
        "attack_code": np.asarray(codes, dtype=np.int64),
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
        "pred_real": pred_real,
        "correct_binary": pred_real == (np.asarray(y3) == 0),
        "pred_class_3": np.argmax(logits, axis=1),
        "correct_3class": np.argmax(logits, axis=1) == np.asarray(y3),
    })
    return result

def eval_spatial(model, loader, device, amp):
    model.eval()
    logits_all, y_all, keys_all, codes_all = [], [], [], []
    with torch.no_grad():
        for x, y, keys, codes in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
            codes_all.extend(np.asarray(codes, dtype=np.int64).tolist())
    if not logits_all:
        raise RuntimeError("Spatial evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    labels = np.asarray(y_all, dtype=np.int64)
    codes = np.asarray(codes_all, dtype=np.int64)
    if len(logits) != len(labels) or len(keys_all) != len(labels) or len(codes) != len(labels):
        raise RuntimeError("Spatial evaluation outputs have inconsistent sample counts")
    return logits, labels, list(map(str, keys_all)), codes
def eval_lcc_spatial(model, loader, device, amp, lcc_frame, threshold):
    model.eval()
    logits_all, y_all, keys_all = [], [], []
    with torch.no_grad():
        for x, y, keys, _status in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
    if not logits_all:
        raise RuntimeError("Common LCC VALID evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    y = np.asarray(y_all, dtype=np.int64)
    keys = list(map(str, keys_all))
    expected_paths = lcc_frame.loc[lcc_frame["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    assert_ordered_keys(keys, expected_paths, "Stage 1 LCC")
    if len(logits) != len(y) or len(keys) != len(y):
        raise RuntimeError("Stage 1 LCC logits/labels/keys have inconsistent counts")
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    scored = pd.DataFrame({
        "path": keys, "label": y,
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
    })
    full = lcc_frame[["path", "label", "status"]].merge(
        scored, on=["path", "label"], how="left", validate="one_to_one"
    )
    full["dataset_role"] = "LCC-FASD external-dev / cross-domain stress set"
    valid = full["status"].astype(str).eq("VALID")
    if full.loc[valid, "pad_logit_score"].isna().any() or int(valid.sum()) != len(keys):
        raise RuntimeError("Stage 1 inference failed to score the exact common LCC VALID set")
    full["pred_real"] = np.nan
    full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)
    full["pred_real"] = full["pred_real"].where(valid, np.nan)
    return logits, y, keys, full
def trainable_parameter_count(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))

def write_zip(folder, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(Path(folder).rglob("*")):
            if path.is_file() and path.resolve() != Path(zip_path).resolve():
                archive.write(path, arcname=Path(Path(folder).name) / path.relative_to(folder))


def state_dict_sha256(state):
    h=hashlib.sha256()
    for key in sorted(state):
        value=state[key].detach().cpu().contiguous()
        h.update(key.encode('utf-8'));h.update(str(value.dtype).encode('ascii'))
        h.update(str(tuple(value.shape)).encode('ascii'));h.update(value.numpy().tobytes())
    return h.hexdigest()


def read_rgb_crop_mnv3_27(image_path, record, domain, key, augment=False):
    # Validate frozen bbox/status provenance, but intentionally ignore E1's stored 1.55/1.50 crop factor.
    validate_bbox_record(record,domain,key)
    bgr=cv2.imread(str(image_path),cv2.IMREAD_COLOR)
    if bgr is None:raise FileNotFoundError(image_path)
    box=bbox_jitter(record) if augment else record['bbox_xyxy']
    crop=runtime_crop_bgr(bgr,box,MNV3_CROP_SCALE)
    rgb=cv2.cvtColor(crop,cv2.COLOR_BGR2RGB)
    if augment:rgb=apply_e1_mild_augmentation(rgb)
    return resize_letterbox_rgb(rgb,INPUT_SIZE)


def save_crop_context_gallery(resources,root,out_path):
    rng=np.random.default_rng(SEED)
    fig,axes=plt.subplots(3,4,figsize=(16,12))
    for cls in range(3):
        pool=np.where(resources['train_labels']==cls)[0]
        idx=int(rng.choice(pool));key=str(resources['train_keys'][idx])
        rec=resources['bbox_cache'][key]
        image=cv2.imread(str(Path(root)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(Path(root)/key)
        bbox=rec['bbox_xyxy'];x1,y1,x2,y2=map(int,bbox)
        marked=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
        cv2.rectangle(marked,(x1,y1),(x2,y2),(255,0,0),2)
        old_factor=float(rec['crop_factor'])
        old=resize_letterbox_rgb(cv2.cvtColor(runtime_crop_bgr(image,bbox,old_factor),cv2.COLOR_BGR2RGB),INPUT_SIZE)
        wide=resize_letterbox_rgb(cv2.cvtColor(runtime_crop_bgr(image,bbox,MNV3_CROP_SCALE),cv2.COLOR_BGR2RGB),INPUT_SIZE)
        model_input=rgb_tensor(wide)
        final=(model_input*torch.as_tensor(CELEBA_STD)[:,None,None]+torch.as_tensor(CELEBA_MEAN)[:,None,None]).clamp(0,1).permute(1,2,0).numpy()
        panels=[marked,old,wide,final]
        titles=['Raw + frozen bbox',f'Old context {old_factor:.2f}x','New context 2.70x','Final MNV3 input 224x224']
        for ax,panel,title in zip(axes[cls],panels,titles):
            ax.imshow(panel);ax.set_title(f'{title} | class={cls}');ax.axis('off')
    fig.tight_layout();fig.savefig(out_path,dpi=130);plt.close(fig)


In [ ]:
import re
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BANDS = ('LOW', 'MID', 'HIGH')

class CSMRTrainRGBDataset(CelebAMicroDataset):
    def __init__(self, *args):
        super().__init__(*args, train=True)
    def __getitem__(self, i):
        key = self.keys[i]
        rgb = read_rgb_crop_mnv3_27(self.root / key, self.records[key], 'celeba', key, augment=True)
        x = torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1))).float() / 255.0
        return x, int(self.labels[i]), key, int(self.codes[i])

def normalize_unit(x):
    mean = torch.as_tensor(CELEBA_MEAN, device=x.device, dtype=x.dtype)[None, :, None, None]
    std = torch.as_tensor(CELEBA_STD, device=x.device, dtype=x.dtype)[None, :, None, None]
    return (x - mean) / std

def radial_masks(size=INPUT_SIZE, device='cpu'):
    y = torch.arange(size, device=device, dtype=torch.float32) - size // 2
    yy, xx = torch.meshgrid(y, y, indexing='ij')
    radius = torch.sqrt(xx.square() + yy.square()) / math.sqrt(2 * (size // 2) ** 2)
    centers = torch.as_tensor(CSMR_BAND_CENTERS, dtype=torch.float32, device=device)
    masks = torch.exp(-((radius[None] - centers[:, None, None]) ** 2) /
                      (2 * CSMR_BAND_SIGMA ** 2))
    if masks.shape != (3, size, size) or not torch.isfinite(masks).all() or not ((masks >= 0) & (masks <= 1)).all():
        raise RuntimeError('Invalid CSMR radial masks')
    if len(set(map(float, CSMR_BAND_CENTERS))) != 3:
        raise RuntimeError('CSMR band centers must differ')
    return masks, radius

def spectral_views(x, gains, masks):
    if x.ndim != 4 or x.shape[1:] != (3, INPUT_SIZE, INPUT_SIZE) or not torch.isfinite(x).all():
        raise ValueError('CSMR input must be finite [B,3,224,224]')
    x = x.float()
    gains = gains.to(device=x.device, dtype=torch.float32).reshape(-1)
    if len(gains) != len(x) or not ((gains >= 0) & (gains <= 1)).all():
        raise ValueError('Invalid gains')
    spectrum = torch.fft.fftshift(torch.fft.fft2(x, norm='ortho'), dim=(-2, -1))
    gain_fields = 1 - (1 - gains[:, None, None, None]) * masks[None]
    if CSMR_PRESERVE_DC:
        gain_fields[:, :, INPUT_SIZE // 2, INPUT_SIZE // 2] = 1.0
    perturbed = spectrum[:, None] * gain_fields[:, :, None]
    raw = torch.fft.ifft2(torch.fft.ifftshift(perturbed, dim=(-2, -1)), norm='ortho').real
    clipped = ((raw < 0) | (raw > 1)).float().mean(dim=(2, 3, 4))
    views = raw.clamp(0, 1)
    if not torch.isfinite(views).all():
        raise RuntimeError('Non-finite CSMR intervention')
    return views, gain_fields, clipped

def aligned_margin(logits, labels):
    d = binary_score(logits.float())
    labels = torch.as_tensor(labels, device=d.device)
    if not torch.isin(labels, torch.tensor([0, 1, 2], device=d.device)).all():
        raise ValueError('Invalid PAD label')
    return torch.where(labels == 0, d, -d)

def harmful_margin_loss(clean_margin, worst_margin):
    drop = clean_margin.detach() - worst_margin
    harmful = drop > 0
    per = F.smooth_l1_loss(worst_margin, clean_margin.detach(), reduction='none')
    return (per * harmful.float()).mean(), harmful, drop

def selection_by_margins(margins):
    if margins.ndim != 2 or margins.shape[1] != 3 or not torch.isfinite(margins).all():
        raise ValueError('Expected finite [B,3] margins')
    return margins.argmin(dim=1)

def select_worst_view(model, base, labels, masks, amp, chunk_size):
    if chunk_size < 1:
        raise ValueError('Selection chunk size must be positive')
    was_training = model.training
    model.eval()
    chosen, indexes, all_gains, all_clips = [], [], [], []
    try:
        with torch.no_grad():
            for start in range(0, len(base), chunk_size):
                x = base[start:start+chunk_size]
                y = labels[start:start+chunk_size]
                gains = torch.empty(len(x), device=x.device).uniform_(*CSMR_GAIN_RANGE)
                views, _, clips = spectral_views(x, gains, masks)
                margins = []
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=amp):
                        logits = model(normalize_unit(views[:, band]))
                    margins.append(aligned_margin(logits, y))
                indexes_chunk = selection_by_margins(torch.stack(margins, dim=1))
                ix = torch.arange(len(x), device=x.device)
                chosen.append(views[ix, indexes_chunk].detach())
                indexes.append(indexes_chunk.detach())
                all_gains.append(gains.detach())
                all_clips.append(clips[ix, indexes_chunk].detach())
    finally:
        model.train(was_training)
    return tuple(torch.cat(a) for a in (chosen, indexes, all_gains, all_clips))

def strict_json_load(path):
    def reject(value):
        raise ValueError('Non-finite JSON constant: ' + value)
    return json.loads(Path(path).read_text(), parse_constant=reject)

def strict_json_write(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, allow_nan=False) + '\n')

def load_stage1_reference(resources):
    path = find_unique_file('micro_e1_v1_summary.json', explicit_dir=STAGE1_RESULT_DIR)
    summary = strict_json_load(path)
    if summary.get('schema') != 'pad_micro_stage1_summary_v1':
        raise RuntimeError('Stage 1 baseline summary schema mismatch')
    if summary.get('resource_hashes') != resources['fingerprints']:
        raise RuntimeError('Stage 0/Stage 1 resource hash mismatch')
    digest = summary.get('checkpoint_sha256', '')
    if not isinstance(digest, str) or not re.fullmatch(r'[0-9a-f]{64}', digest):
        raise RuntimeError('Invalid Stage 1 checkpoint SHA256')
    for name in ('micro_e1_v1_meta.json', 'micro_e1_v1_config.json'):
        candidate = Path(path).parent / name
        if candidate.exists():
            companion = strict_json_load(candidate)
            if companion.get('checkpoint_sha256') != digest or companion.get('resource_hashes') != resources['fingerprints']:
                raise RuntimeError('Stage 1 companion SHA/resource mismatch: ' + name)
    candidate = Path(path).parent / 'micro_e1_v1_best.pth'
    if candidate.exists() and sha256_file(candidate) != digest:
        raise RuntimeError('Stage 1 checkpoint file SHA mismatch')
    for split, n in (('val', 2000), ('lcc', len(resources['lcc_valid_paths']))):
        m = summary.get(split, {})
        if int(m.get('n_scored', -1)) != n:
            raise RuntimeError('Stage 1 ' + split + ' count mismatch')
        for metric in ('acer', 'auc', 'apcer', 'bpcer', 'hter'):
            if not math.isfinite(float(m[metric])):
                raise RuntimeError('Non-finite Stage 1 metric')
        if float(m['locked_logit_threshold']) != float(summary['locked_validation_logit_threshold']):
            raise RuntimeError('Stage 1 threshold mismatch')
    return summary, path

def intervention_sanity(masks, device):
    gen = torch.Generator(device=device).manual_seed(SEED)
    x = torch.rand((2, 3, INPUT_SIZE, INPUT_SIZE), generator=gen, device=device)
    gain = torch.tensor([0.65, 0.90], device=device)
    views, fields, clips = spectral_views(x, gain, masks)
    assert views.shape == (2, 3, 3, INPUT_SIZE, INPUT_SIZE)
    assert torch.all(fields[:, :, INPUT_SIZE//2, INPUT_SIZE//2] == 1)
    unity, _, _ = spectral_views(x, torch.ones(2, device=device), masks)
    assert torch.max(torch.abs(unity[:, 0] - x)).item() < 1e-5
    ft = torch.fft.fftshift(torch.fft.fft2(x, norm='ortho'), dim=(-2,-1))
    modified = ft[:, None] * fields[:, :, None]
    valid = ft.abs()[:, None] > 1e-5
    phase_error = (torch.angle(modified) - torch.angle(ft[:, None])).abs()
    phase_error = torch.minimum(phase_error, 2*math.pi-phase_error)
    assert phase_error.expand_as(valid.expand_as(modified))[valid.expand_as(modified)].max().item() < 1e-4
    report = {}
    for i, band in enumerate(BANDS):
        diff = (views[:, i] - x).abs()
        report[band] = {'mean_absolute_pixel_difference': float(diff.mean()),
                        'max_absolute_pixel_difference': float(diff.max()),
                        'clipped_pixel_fraction': float(clips[:, i].mean())}
        if report[band]['clipped_pixel_fraction'] > 0.05:
            print('WARNING: clipping exceeds 5% for', band)
    return report

def save_gallery(train_ds, masks, out_dir):
    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    radius = radial_masks(device='cpu')[1].numpy()
    ax[0].imshow(radius, cmap='viridis', vmin=0, vmax=1); ax[0].set_title('r: center=0, corner≈1')
    for i, band in enumerate(BANDS):
        ax[i+1].imshow(masks[i].detach().cpu().numpy(), cmap='viridis', vmin=0, vmax=1)
        ax[i+1].set_title(f'{band} c={CSMR_BAND_CENTERS[i]}')
    for a in ax: a.axis('off')
    fig.tight_layout(); fig.savefig(out_dir/'csmr_v1_frequency_masks.png', dpi=130); plt.close(fig)
    rng = np.random.default_rng(SEED)
    indices = []
    for cls in range(3):
        pool = np.where(train_ds.labels == cls)[0]
        if len(pool) < 2: raise RuntimeError('Gallery requires 2 examples per class')
        indices.extend(rng.choice(pool, 2, replace=False).tolist())
    fig, ax = plt.subplots(6, 4, figsize=(16, 20))
    for row, idx in enumerate(indices):
        random.seed(SEED + idx); np.random.seed(SEED + idx)
        x, label, key, _ = train_ds[idx]
        gain = float(rng.uniform(*CSMR_GAIN_RANGE))
        views, _, _ = spectral_views(x[None], torch.tensor([gain]), masks.cpu())
        for col, item in enumerate([x, *list(views[0])]):
            ax[row,col].imshow(item.permute(1,2,0).numpy())
            ax[row,col].set_title(f'{["Clean", *BANDS][col]} | class={label} | g={gain:.3f}\n{key[-45:]}')
            ax[row,col].axis('off')
    fig.tight_layout(); fig.savefig(out_dir/'csmr_v1_intervention_gallery.png', dpi=110); plt.close(fig)


In [ ]:
from datetime import datetime, timezone
from sklearn.model_selection import train_test_split
from IPython.display import FileLink, display

RUN_STARTED = time.monotonic()

def progress(section, message):
    stamp = datetime.now(timezone.utc).isoformat(timespec='seconds')
    print(f'[{stamp} | +{time.monotonic()-RUN_STARTED:,.1f}s | {section}] {message}', flush=True)

def gpu_usage():
    if not torch.cuda.is_available():
        return 'CUDA unavailable'
    return (f'GPU {torch.cuda.get_device_name(0)} | allocated '
            f'{torch.cuda.memory_allocated()/2**20:.0f} MiB | '
            f'reserved {torch.cuda.memory_reserved()/2**20:.0f} MiB | '
            f'peak {torch.cuda.max_memory_allocated()/2**20:.0f} MiB')

def check_fixed_contract():
    if (SEED!=100 or BATCH_SIZE not in (128,64) or MAX_EPOCHS!=12 or
        EARLY_STOP_PATIENCE!=3 or EARLIEST_EARLY_STOP_EPOCH!=6 or
        MNV3_CROP_SCALE!=2.7 or BACKBONE_LR!=1e-5 or HEAD_LR!=1e-4 or
        WEIGHT_DECAY!=1e-4 or LABEL_SMOOTHING!=.1 or GRAD_CLIP!=5.0 or
        CSMR_WARMUP_EPOCHS!=1 or CSMR_LAMBDA!=.20 or
        CSMR_BAND_CENTERS!=[.15,.45,.75] or CSMR_BAND_SIGMA!=.10 or
        CSMR_GAIN_RANGE!=[.65,.90] or CSMR_PRESERVE_DC is not True or
        SAVE_INTERVENTION_GALLERY is not True or SELECTION_CHUNK_SIZE<1):
        raise RuntimeError('Fixed crop27 seed100 recipe or frozen CSMR-v1 contract changed')


def make_optimizer(model):
    return torch.optim.AdamW([
        {'params': list(model.features.parameters())+list(model.avgpool.parameters()), 'lr':BACKBONE_LR},
        {'params': list(model.projection.parameters())+list(model.classifier.parameters()), 'lr':HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

def model_state_cpu(model):
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def evaluate_val(model, loader, resources, device, use_amp):
    logits,y,keys,codes = eval_spatial(model,loader,device,use_amp)
    assert_ordered_keys(keys,resources['val_keys'],'frozen micro-Val')
    threshold = threshold_from_val(y,binary_score(torch.as_tensor(logits)).cpu().numpy())
    metrics,_ = metrics_celeba(y,logits,threshold)
    return metrics,threshold,(logits,y,keys,codes)

def train_one(name, csmr, resources, celeba_root, initial_state, val_loader, device, use_amp, masks, output_root):
    progress(name,'Seeding training RNG=100 before model and DataLoader construction')
    seed_everything(SEED)
    train_ds = CSMRTrainRGBDataset(resources['train_keys'],resources['train_labels'],
        resources['train_codes'],celeba_root,resources['bbox_cache'],resources['stage0_dir'])
    out_dir=output_root/('csmr' if csmr else 'vanilla')
    out_dir.mkdir()
    if csmr:
        sanity=intervention_sanity(masks,device)
        progress(name,f'FFT sanity: {sanity}')
        save_gallery(train_ds,masks,out_dir)
        progress(name,'Saved mask figure and 6-sample intervention gallery')
        seed_everything(SEED)  # Same model/loader initialization despite gallery RNG.
    else:
        sanity=None
    model=SpatialE1(None).to(device)
    model.load_state_dict(initial_state,strict=True)
    branch_initial_sha=state_dict_sha256(model.state_dict())
    if branch_initial_sha!=initial_state_sha256:
        raise RuntimeError(f'{name} initial state differs from canonical SHA')
    progress(name,f'Canonical ImageNet-initialized state SHA={branch_initial_sha}; params={trainable_parameter_count(model)}; {gpu_usage()}')
    loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,drop_last=False,
        num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=NUM_WORKERS>0)
    progress(name,f'Train batches={len(loader)}; logging every {max(1,len(loader)//18)} batches')
    optimizer=make_optimizer(model)
    criterion=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    history=[]; diagnostics=[]; class_rows=[]
    best_acer=float('inf'); best_epoch=None; best_state=None
    try:
        for epoch in range(1,MAX_EPOCHS+1):
            epoch_started=time.monotonic();model.train()
            progress(name,f'Epoch {epoch}/{MAX_EPOCHS} start; {gpu_usage()}')
            totals=[]; cls_losses=[]; margin_losses=[]
            band_counts=torch.zeros(3,dtype=torch.int64)
            class_counts=torch.zeros((3,3),dtype=torch.int64)
            class_harmful=torch.zeros(3,dtype=torch.int64)
            class_drop=torch.zeros(3,dtype=torch.float64)
            class_seen=torch.zeros(3,dtype=torch.int64)
            harmful_n=0; harmful_drop_sum=clean_margin_sum=worst_margin_sum=gain_sum=clip_sum=0.
            csmr_n=0
            for batch_idx,(base,labels,_keys,_codes) in enumerate(loader,1):
                base=base.to(device,non_blocking=True)
                labels=labels.to(device,non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                if csmr and epoch>CSMR_WARMUP_EPOCHS:
                    selected,band_idx,gains,clips=select_worst_view(
                        model,base,labels,masks,use_amp,SELECTION_CHUNK_SIZE)
                    model.train()
                    combined=torch.cat([normalize_unit(base),normalize_unit(selected)],dim=0)
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        logits=model(combined)
                        clean_logits,worst_logits=logits.chunk(2)
                        cls_loss=.5*criterion(clean_logits,labels)+.5*criterion(worst_logits,labels)
                        clean_margin=aligned_margin(clean_logits,labels)
                        worst_margin=aligned_margin(worst_logits,labels)
                        margin_loss,harmful,drop=harmful_margin_loss(clean_margin,worst_margin)
                        loss=cls_loss+CSMR_LAMBDA*margin_loss
                    n=len(labels);csmr_n+=n
                    gain_sum+=float(gains.sum().cpu());clip_sum+=float(clips.sum().cpu())
                    harmful_n+=int(harmful.sum().cpu())
                    harmful_drop_sum+=float(drop.clamp_min(0).sum().detach().cpu())
                    clean_margin_sum+=float(clean_margin.sum().detach().cpu())
                    worst_margin_sum+=float(worst_margin.sum().detach().cpu())
                    for band in range(3):
                        band_counts[band]+=int((band_idx==band).sum().cpu())
                    for cls in range(3):
                        mask=labels==cls
                        class_seen[cls]+=int(mask.sum().cpu())
                        class_harmful[cls]+=int((harmful&mask).sum().cpu())
                        class_drop[cls]+=float(drop.clamp_min(0)[mask].sum().detach().cpu())
                        for band in range(3):
                            class_counts[cls,band]+=int((mask&(band_idx==band)).sum().cpu())
                else:
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        clean_logits=model(normalize_unit(base))
                        cls_loss=criterion(clean_logits,labels)
                        margin_loss=cls_loss.detach()*0
                        loss=cls_loss
                if not torch.isfinite(loss):
                    raise RuntimeError(f'{name} epoch {epoch} batch {batch_idx}: non-finite loss')
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                scaler.step(optimizer);scaler.update()
                totals.append(float(loss.detach().cpu()))
                cls_losses.append(float(cls_loss.detach().cpu()))
                margin_losses.append(float(margin_loss.detach().cpu()))
                if batch_idx==1 or batch_idx%max(1,len(loader)//18)==0 or batch_idx==len(loader):
                    progress(name,f'E{epoch} batch {batch_idx}/{len(loader)} loss={totals[-1]:.4f} mean={np.mean(totals):.4f} {gpu_usage()}')
            val_metrics,threshold,_=evaluate_val(model,val_loader,resources,device,use_amp)
            row={'epoch':epoch,'train_loss':float(np.mean(totals)),
                 'val_acer':val_metrics['acer'],'val_auc':val_metrics['auc'],
                 'val_apcer':val_metrics['apcer'],'val_bpcer':val_metrics['bpcer'],
                 'val_threshold':float(threshold),'seconds':time.monotonic()-epoch_started}
            history.append(row)
            progress(name,f'Epoch {epoch} end: {row}')
            if csmr:
                d={'epoch':epoch,'lambda':CSMR_LAMBDA if csmr_n else 0.,
                   'train_total_loss':float(np.mean(totals)),
                   'train_cls_loss':float(np.mean(cls_losses)),
                   'train_margin_loss':float(np.mean(margin_losses)) if csmr_n else None}
                for band,count in zip(BANDS,band_counts.tolist()):
                    d[f'worst_{band.lower()}_count']=count if csmr_n else None
                    d[f'worst_{band.lower()}_fraction']=count/csmr_n if csmr_n else None
                d.update({'harmful_fraction':harmful_n/csmr_n if csmr_n else None,
                    'mean_harmful_margin_drop':harmful_drop_sum/harmful_n if harmful_n else None,
                    'mean_clean_aligned_margin':clean_margin_sum/csmr_n if csmr_n else None,
                    'mean_worst_aligned_margin':worst_margin_sum/csmr_n if csmr_n else None,
                    'mean_sampled_gain':gain_sum/csmr_n if csmr_n else None,
                    'mean_clip_fraction':clip_sum/csmr_n if csmr_n else None})
                diagnostics.append(d)
                if csmr_n:
                    if max(d[f'worst_{b.lower()}_fraction'] for b in BANDS)>.95:
                        progress(name,'WARNING: one band selected for >95% of samples')
                    if d['mean_clip_fraction']>.05:
                        progress(name,'WARNING: mean clipped pixel fraction >5%')
                for cls,cls_name in enumerate(('Real','Physical','Digital')):
                    n=int(class_seen[cls])
                    cr={'epoch':epoch,'class':cls_name,'n':n if csmr_n else None,
                        'harmful_fraction':int(class_harmful[cls])/n if n else None,
                        'mean_margin_drop':float(class_drop[cls])/n if n else None}
                    for band in range(3):
                        cr[f'worst_{BANDS[band].lower()}_count']=int(class_counts[cls,band]) if n else None
                        cr[f'worst_{BANDS[band].lower()}_fraction']=int(class_counts[cls,band])/n if n else None
                    class_rows.append(cr)
            if val_metrics['acer']<best_acer:
                best_acer=val_metrics['acer'];best_epoch=epoch;best_state=model_state_cpu(model)
                progress(name,f'NEW BEST epoch={epoch} Val ACER={best_acer:.6f}')
            elif epoch>=EARLIEST_EARLY_STOP_EPOCH and epoch-best_epoch>=EARLY_STOP_PATIENCE:
                progress(name,'Early stop on frozen micro-Val ACER');break
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('CUDA OOM: restart clean and change only BATCH_SIZE to 64') from exc
    if best_state is None:raise RuntimeError(f'{name}: no checkpoint selected')
    last_state=model_state_cpu(model)
    model.load_state_dict(best_state)
    final_val,locked,outputs=evaluate_val(model,val_loader,resources,device,use_amp)
    logits,y,keys,codes=outputs
    best_path=out_dir/'best.pth';last_path=out_dir/'checkpoint_last.pth'
    torch.save({'state_dict':best_state,'epoch':best_epoch,'val_acer':best_acer},best_path)
    torch.save({'state_dict':last_state,'epoch':epoch},last_path)
    checkpoint_sha=sha256_file(best_path)
    pd.DataFrame(history).to_csv(out_dir/'training_history.csv',index=False)
    if csmr:
        pd.DataFrame(diagnostics).to_csv(out_dir/'training_diagnostics.csv',index=False)
        pd.DataFrame(class_rows).to_csv(out_dir/'class_diagnostics.csv',index=False)
    make_prediction_frame(keys,y,codes,logits,locked).to_csv(out_dir/'val_predictions.csv',index=False)
    config={'schema':'csmr_seed100_experiment_config_v1','name':name,'training_seed':SEED,
        'development_manifest_name':'micro_manifest_v1_seed42.npz',
        'resource_hashes':resources['fingerprints'],'architecture':E1_ARCHITECTURE,
        'preprocessing':'frozen Stage-0 bbox; crop 2.7; RGB 224; existing letterbox and CelebA normalization; gamma off',
        'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'patience':EARLY_STOP_PATIENCE,
        'backbone_lr':BACKBONE_LR,'head_lr':HEAD_LR,'weight_decay':WEIGHT_DECAY,
        'label_smoothing':LABEL_SMOOTHING,'grad_clip':GRAD_CLIP,'amp':use_amp,
        'crop_scale':MNV3_CROP_SCALE,'input_size':INPUT_SIZE,'channel_order':'RGB','initial_state_sha256':branch_initial_sha,
        'csmr':csmr,'csmr_settings':({'warmup_epochs':CSMR_WARMUP_EPOCHS,
            'lambda':CSMR_LAMBDA,'centers':CSMR_BAND_CENTERS,'sigma':CSMR_BAND_SIGMA,
            'gain_range':CSMR_GAIN_RANGE,'preserve_dc':CSMR_PRESERVE_DC} if csmr else None)}
    strict_json_write(out_dir/'config.json',config)
    meta={'schema':'csmr_seed100_experiment_meta_v1','name':name,
        'checkpoint_file':best_path.name,'checkpoint_sha256':checkpoint_sha,
        'selected_epoch':int(best_epoch),'locked_val_threshold':float(locked),
        'threshold_source':'frozen CelebA micro-Val only',
        'val':final_val,'resource_hashes':resources['fingerprints'],
        'parameter_count':int(sum(p.numel() for p in model.parameters())),
        'stage1_checkpoint_loaded':False,'initial_state_sha256':branch_initial_sha}
    strict_json_write(out_dir/'meta.json',meta)
    model.cpu()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    progress(name,f'Frozen best epoch={best_epoch}, Val ACER={final_val["acer"]:.6f}, threshold={locked:.6f}, SHA={checkpoint_sha}; {gpu_usage()}')
    return {'name':name,'model':model,'out_dir':out_dir,'val':final_val,
        'locked_threshold':float(locked),'selected_epoch':int(best_epoch),
        'checkpoint_sha256':checkpoint_sha,'initial_state_sha256':branch_initial_sha,'diagnostics':diagnostics,'sanity':sanity}

def evaluate_lcc(result,lcc_loader,resources,device,use_amp):
    name=result['name'];progress(name,'LCC external-development evaluation begins')
    model=result['model'].to(device)
    logits,y,keys,pred=eval_lcc_spatial(model,lcc_loader,device,use_amp,
        resources['lcc_df'],result['locked_threshold'])
    score=binary_score(torch.as_tensor(logits)).cpu().numpy()
    candidate_n=len(resources['lcc_df'])
    metrics=metrics_binary(y,score,result['locked_threshold'],
        coverage=len(y)/candidate_n,candidate_n=candidate_n)
    pred['locked_threshold']=result['locked_threshold']
    pred.to_csv(result['out_dir']/'lcc_predictions.csv',index=False)
    result['lcc']=metrics
    model.cpu();torch.cuda.empty_cache()
    progress(name,f'LCC HTER={metrics["hter"]:.6f}, AUC={metrics["auc"]:.6f}, coverage={metrics["detector_coverage"]:.5f}')
    return result

def freeze_gate(vanilla,csmr):
    delta={'val_acer':csmr['val']['acer']-vanilla['val']['acer'],
        'val_auc':csmr['val']['auc']-vanilla['val']['auc'],
        'lcc_apcer':csmr['lcc']['apcer']-vanilla['lcc']['apcer'],
        'lcc_bpcer':csmr['lcc']['bpcer']-vanilla['lcc']['bpcer'],
        'lcc_hter':csmr['lcc']['hter']-vanilla['lcc']['hter'],
        'lcc_auc':csmr['lcc']['auc']-vanilla['lcc']['auc']}
    primary=delta['lcc_auc']>=.015 and delta['val_acer']<=.010
    secondary=(delta['lcc_hter']<=-.015 and delta['lcc_auc']>=0 and delta['val_acer']<=.010)
    gate='PRIMARY_AUC_GATE' if primary else 'SECONDARY_HTER_GATE' if secondary else 'NONE'
    return ('PASS' if primary or secondary else 'FAIL'),gate,delta

def paired_table(vanilla,csmr,split):
    if split=='development':
        rows=[('Val ACER','val','acer'),('Val AUC','val','auc'),
              ('LCC APCER','lcc','apcer'),('LCC BPCER','lcc','bpcer'),
              ('LCC HTER','lcc','hter'),('LCC AUC','lcc','auc')]
    else:
        rows=[('Test APCER','test','apcer'),('Test BPCER','test','bpcer'),
              ('Test ACER','test','acer'),('Test AUC','test','auc')]
    text=['| Metric | Vanilla seed100 | CSMR seed100 | Delta (CSMR - Vanilla) |',
          '|---|---:|---:|---:|']
    for label,field,metric in rows:
        a=vanilla[field][metric];b=csmr[field][metric]
        text.append(f'| {label} | {a:.6f} | {b:.6f} | {b-a:+.6f} |')
    return '\n'.join(text)


In [ ]:
# Development only. The official Test manifest/images are not read here.
check_fixed_contract()
progress('PREFLIGHT','MNV3-S crop 2.7x paired recipe | seed 100 | frozen Stage-0 seed42 manifest')
if not torch.cuda.is_available():raise RuntimeError('Kaggle GPU accelerator is required')
device=torch.device('cuda');use_amp=bool(AMP)
progress('PREFLIGHT',f'GPU/AMP: {gpu_usage()} | AMP={use_amp}')
resources=load_stage0_resources()
stage1_reference,stage1_path=load_stage1_reference(resources)
celeba_root=find_celeba_root();lcc_root=find_lcc_root(resources['stage0_dir']/'lcc_external_dev_manifest.csv')
assert_raw_image_paths(resources,celeba_root,lcc_root)
progress('PREFLIGHT',f'Train={len(resources["train_keys"])} Val={len(resources["val_keys"])} LCC valid={len(resources["lcc_valid_paths"])}')
loader_args={'batch_size':BATCH_SIZE,'shuffle':False,'drop_last':False,'num_workers':NUM_WORKERS,
    'pin_memory':True,'persistent_workers':NUM_WORKERS>0}
val_ds=CelebAMicroDataset(resources['val_keys'],resources['val_labels'],resources['val_codes'],
    celeba_root,resources['bbox_cache'],resources['stage0_dir'],train=False)
lcc_ds=LCCValidDataset(resources['lcc_df'],lcc_root,resources['lcc_cache'],resources['stage0_dir'])
val_loader=DataLoader(val_ds,**loader_args);lcc_loader=DataLoader(lcc_ds,**loader_args)
output_root=Path(OUTPUT_ROOT)
if output_root.exists():raise RuntimeError(f'Output collision at {output_root}; use a clean Kaggle output directory')
output_root.mkdir(parents=True)
comparison_dir=output_root/'comparison';comparison_dir.mkdir()
strict_json_write(comparison_dir/'resource_fingerprints.json',resources['fingerprints'])
save_crop_context_gallery(resources,celeba_root,comparison_dir/'crop_context_gallery.png')
progress('PREFLIGHT','Saved raw/bbox, old crop, 2.7x crop and final 224x224 RGB gallery')
masks,_=radial_masks(device=device)
imagenet_state=load_imagenet_weights(IMAGENET_WEIGHTS_PATH)
seed_everything(SEED)
canonical_model=SpatialE1(imagenet_state)
initial_state=model_state_cpu(canonical_model)
initial_state_sha256=state_dict_sha256(initial_state)
del canonical_model
strict_json_write(comparison_dir/'initialization_fingerprint.json',{
    'schema':'mnv3s_crop27_seed100_initialization_v1','seed':SEED,
    'initial_state_sha256':initial_state_sha256,'imageNet_source':'torchvision DEFAULT or explicit configured compatible state_dict',
    'resource_hashes':resources['fingerprints']})
progress('PREFLIGHT',f'Canonical ImageNet MNV3-S initial state SHA={initial_state_sha256}')
vanilla=train_one('mnv3s_crop27_vanilla_seed100',False,resources,celeba_root,initial_state,
    val_loader,device,use_amp,masks,output_root)
csmr=train_one('mnv3s_crop27_csmr_seed100',True,resources,celeba_root,initial_state,
    val_loader,device,use_amp,masks,output_root)
if vanilla['initial_state_sha256']!=csmr['initial_state_sha256'] or vanilla['initial_state_sha256']!=initial_state_sha256:
    raise RuntimeError('Vanilla and CSMR did not start from the same canonical state')
progress('DEVELOPMENT','Both Val-selected checkpoints and locked thresholds saved; now evaluating common LCC once')
vanilla=evaluate_lcc(vanilla,lcc_loader,resources,device,use_amp)
csmr=evaluate_lcc(csmr,lcc_loader,resources,device,use_amp)
paired_delta={k:csmr['lcc'][k]-vanilla['lcc'][k] for k in ('apcer','bpcer','hter','auc')}
paired_delta['val_acer']=csmr['val']['acer']-vanilla['val']['acer']
paired_delta['val_auc']=csmr['val']['auc']-vanilla['val']['auc']
paired_rows=[]
for label,split,metric in [('Val ACER','val','acer'),('Val AUC','val','auc'),
    ('LCC APCER','lcc','apcer'),('LCC BPCER','lcc','bpcer'),('LCC HTER','lcc','hter'),('LCC AUC','lcc','auc')]:
    paired_rows.append({'metric':label,'vanilla':vanilla[split][metric],
        'csmr':csmr[split][metric],'delta_csmr_minus_vanilla':csmr[split][metric]-vanilla[split][metric]})
pd.DataFrame(paired_rows).to_csv(comparison_dir/'paired_metrics.csv',index=False)
print('| Metric | MNV3-S 2.7x Vanilla | MNV3-S 2.7x CSMR | Delta |\n|---|---:|---:|---:|')
for row in paired_rows:print(f'| {row["metric"]} | {row["vanilla"]:.6f} | {row["csmr"]:.6f} | {row["delta_csmr_minus_vanilla"]:+.6f} |')
progress('DEVELOPMENT','Val/LCC comparison saved. Any Test evaluation is a later optional descriptive step.')


In [ ]:
def _score_pair(summary):
    result=[]
    for key,label in [('vanilla','Vanilla'),('csmr','CSMR')]:
        branch=summary.get(key,{})
        val=branch.get('val',{});lcc=branch.get('lcc',{})
        result.append({'recipe':label,'val_acer':val.get('acer'),'lcc_hter':lcc.get('hter'),'lcc_auc':lcc.get('auc')})
    return result

def _fmt(x):return 'NA' if x is None else f'{float(x):.6f}'
def _summary_rows(label,summary):
    if not isinstance(summary,dict):return []
    pairs=_score_pair(summary)
    output=[]
    for item in pairs:
        is_csmr=item['recipe']=='CSMR'
        backbone='MiniFASNetV2' if label=='MiniFASNetV2' else 'MNV3-S'
        recipe=f'{backbone} {"+ CSMR" if is_csmr else "Vanilla"}'
        crop='2.7x' if backbone=='MiniFASNetV2' or 'crop27' in str(summary.get('schema','')) else '1.55x'
        input_size='80' if backbone=='MiniFASNetV2' else '224'
        pretraining='PAD pretrained' if backbone=='MiniFASNetV2' else 'ImageNet'
        output.append({'recipe':recipe,'crop':crop,'input':input_size,'pretraining':pretraining,
            'val_acer':item['val_acer'],'lcc_hter':item['lcc_hter'],'lcc_auc':item['lcc_auc']})
    return output

old_mnv3=None;mini_reference=None
if OLD_MNV3_SUMMARY_PATH:
    p=Path(OLD_MNV3_SUMMARY_PATH).expanduser()
    if not p.is_file():raise FileNotFoundError(f'Configured old MNV3 summary missing: {p}')
    old_mnv3=strict_json_load(p)
if MINIFASNET_SUMMARY_PATH:
    p=Path(MINIFASNET_SUMMARY_PATH).expanduser()
    if not p.is_file():raise FileNotFoundError(f'Configured MiniFASNet summary missing: {p}')
    mini_reference=strict_json_load(p)
current_summary={'schema':'mnv3s_crop27_csmr_seed100_v1','role':'practical recipe comparison',
    'training_seed':SEED,'crop_scale':MNV3_CROP_SCALE,'input_size':INPUT_SIZE,
    'training_budget':{'max_epochs':MAX_EPOCHS,'patience':EARLY_STOP_PATIENCE,
        'earliest_early_stop_epoch':EARLIEST_EARLY_STOP_EPOCH},
    'resource_hashes':resources['fingerprints'],'initial_state_sha256':initial_state_sha256,
    'vanilla':{'selected_epoch':vanilla['selected_epoch'],'locked_val_threshold':vanilla['locked_threshold'],
        'checkpoint_sha256':vanilla['checkpoint_sha256'],'val':vanilla['val'],'lcc':vanilla['lcc']},
    'csmr':{'selected_epoch':csmr['selected_epoch'],'locked_val_threshold':csmr['locked_threshold'],
        'checkpoint_sha256':csmr['checkpoint_sha256'],'val':csmr['val'],'lcc':csmr['lcc']},
    'delta_csmr_minus_vanilla':paired_delta,'fixed_opened_test2000':None,
    'historical_reference_paths':{'old_mnv3_summary':OLD_MNV3_SUMMARY_PATH or None,
        'minifasnet_summary':MINIFASNET_SUMMARY_PATH or None}}
strict_json_write(comparison_dir/'transfer_summary.json',current_summary)
reference_rows=[]
if old_mnv3 is not None:
    for row in _summary_rows('MNV3-S',old_mnv3):
        row['recipe']=row['recipe'].replace('MNV3-S','MNV3-S old');reference_rows.append(row)
for base,item in [('vanilla',vanilla),('csmr',csmr)]:
    reference_rows.append({'recipe':'MNV3-S new '+('Vanilla' if base=='vanilla' else '+ CSMR'),
        'crop':'2.7x','input':'224','pretraining':'ImageNet','val_acer':item['val']['acer'],
        'lcc_hter':item['lcc']['hter'],'lcc_auc':item['lcc']['auc']})
if mini_reference is not None:reference_rows.extend(_summary_rows('MiniFASNetV2',mini_reference))
reference_df=pd.DataFrame(reference_rows,columns=['recipe','crop','input','pretraining','val_acer','lcc_hter','lcc_auc'])
reference_df.to_csv(comparison_dir/'historical_recipe_comparison.csv',index=False)
summary_table=['| Recipe | Crop | Input | Pretraining | Val ACER | LCC HTER | LCC AUC |',
    '|---|---:|---:|---|---:|---:|---:|']
for row in reference_rows:
    summary_table.append(f'| {row["recipe"]} | {row["crop"]} | {row["input"]} | {row["pretraining"]} | {_fmt(row["val_acer"])} | {_fmt(row["lcc_hter"])} | {_fmt(row["lcc_auc"])} |')
if old_mnv3 is None:summary_table.append('| MNV3-S old Vanilla/CSMR | 1.55x | 224 | ImageNet | NA | NA | NA |')
if mini_reference is None:
    summary_table += ['| MiniFASNetV2 Vanilla | 2.7x | 80 | PAD pretrained | NA | NA | NA |',
        '| MiniFASNetV2 + CSMR | 2.7x | 80 | PAD pretrained | NA | NA | NA |']
def _crop27_make_paired_table(vanilla_result,csmr_result):
    rows=[('Val ACER','val','acer'),('Val AUC','val','auc'),
        ('LCC APCER','lcc','apcer'),('LCC BPCER','lcc','bpcer'),
        ('LCC HTER','lcc','hter'),('LCC AUC','lcc','auc')]
    table=['| Metric | MNV3-S 2.7x Vanilla | MNV3-S 2.7x CSMR | Delta |',
        '|---|---:|---:|---:|']
    for label,split,metric in rows:
        a=vanilla_result[split][metric];b=csmr_result[split][metric]
        table.append(f'| {label} | {a:.6f} | {b:.6f} | {b-a:+.6f} |')
    return '\n'.join(table)

paired_metrics_table=_crop27_make_paired_table(vanilla,csmr)
report=['# MNV3-S 2.7x / 12-epoch CSMR seed-100 recipe comparison','',
    '## Protocol and interpretation','This practical recipe comparison changes both crop context (old 1.55x to 2.7x) and budget (old 6 to 12 epochs). It is not a pure architecture ablation or a crop-only causal test. MNV3-S uses ImageNet pretraining and 224x224 input; MiniFASNetV2 uses PAD pretraining and 80x80 input.','',
    f'Frozen Stage-0 seed42 resource SHA `{resources["fingerprints"]["micro_manifest_sha256"]}`; 5,000 Train / 2,000 Val; training seed 100. Both current branches began at state SHA `{initial_state_sha256}`. Crop uses the same frozen bbox/status records with a separate 2.7x adapter, RGB, 224x224 letterbox, and existing CelebA normalization.','',
    '## New paired Val/LCC results',paired_metrics_table,'',
    '## Historical recipe comparison',*summary_table,'',
    '## Questions','1. **Does the wider-context MNV3 recipe improve over old MNV3?** ']
if old_mnv3 is None:
    report.append('Undetermined until `OLD_MNV3_SUMMARY_PATH` points to the completed old summary. The current run also changes the budget from 6 to 12 epochs, so any difference cannot be attributed to crop alone.')
else:
    old_v=old_mnv3.get('vanilla',{});new_v=current_summary['vanilla']
    old_auc=old_v.get('lcc',{}).get('auc');old_hter=old_v.get('lcc',{}).get('hter')
    if old_auc is None or old_hter is None:report.append('Old summary lacks Vanilla LCC AUC/HTER.')
    else:report.append(f'For Vanilla, LCC AUC delta is {new_v["lcc"]["auc"]-float(old_auc):+.6f} and HTER delta is {new_v["lcc"]["hter"]-float(old_hter):+.6f} (new minus old). Crop and budget changed together.')
report+=['','2. **Does frozen CSMR still help MNV3-S at 2.7x?** '
    f'CSMR-minus-Vanilla LCC AUC={paired_delta["auc"]:+.6f}, LCC HTER={paired_delta["hter"]:+.6f}, Val ACER={paired_delta["val_acer"]:+.6f}. Positive AUC and negative HTER favor CSMR.','',
    '3. **Does MNV3-S 2.7x close the LCC gap to MiniFASNetV2 2.7x?** ']
if mini_reference is None:report.append('Undetermined until `MINIFASNET_SUMMARY_PATH` points to the completed MiniFASNetV2 transfer summary.')
else:
    mv=mini_reference.get('vanilla',{});mc=mini_reference.get('csmr',{})
    gaps=[]
    for era,mnv3_summary in ([('old',old_mnv3)] if isinstance(old_mnv3,dict) else [])+[('new',current_summary)]:
        for name,key,ref in [('Vanilla','vanilla',mv),('CSMR','csmr',mc)]:
            mnv3_branch=mnv3_summary.get(key,{})
            a=mnv3_branch.get('lcc',{}).get('auc');h=mnv3_branch.get('lcc',{}).get('hter')
            ref_a=ref.get('lcc',{}).get('auc');ref_h=ref.get('lcc',{}).get('hter')
            if a is not None and h is not None and ref_a is not None and ref_h is not None:
                gaps.append(f'{era} {name}: MNV3 minus Mini LCC AUC={float(a)-float(ref_a):+.6f}, HTER={float(h)-float(ref_h):+.6f}')
    report.append('; '.join(gaps) if gaps else 'Mini/old MNV3 summaries lack required paired LCC metrics.')
report+=['','4. **Practical backbone candidate.** If MiniFASNetV2 remains better on both LCC AUC and HTER with references supplied, this supports choosing its current recipe as the practical final backbone candidate. This does not establish that architecture alone caused the difference.','',
    '## CSMR diagnostics']
if csmr['diagnostics']:
    d=csmr['diagnostics'][-1]
    report.append('Final epoch LOW/MID/HIGH worst-view fractions: '+
        '/'.join(f'{d[f"worst_{b.lower()}_fraction"]:.3f}' for b in BANDS)+
        f'; harmful fraction={d["harmful_fraction"]:.3f}; mean clean margin={d["mean_clean_aligned_margin"]}; mean worst margin={d["mean_worst_aligned_margin"]}; sampled gain={d["mean_sampled_gain"]:.3f}; clip fraction={d["mean_clip_fraction"]:.4f}.')
report+=['','## Test-2000','Not evaluated in this cell. If both exact prior manifest and bbox-cache paths are configured, the next cell evaluates only those identities after Val/LCC summary persistence.','',
    '## Limits','One paired training seed. Old-to-new MNV3 comparison changes crop and epoch budget together. Cross-backbone comparison also changes architecture, input size, and pretraining. Historical summaries are reference-only; Test is descriptive and cannot affect selection.','']
(comparison_dir/'comparison_report.md').write_text('\n'.join(report))
progress('SUMMARY','Val/LCC summary, historical comparison table, and question-focused report saved before optional Test access')


In [ ]:
# Optional fixed opened Test-2000 descriptive benchmark; no sampling or resampling.
if bool(FIXED_TEST_MANIFEST_PATH)!=bool(FIXED_TEST_BBOX_CACHE_PATH):
    raise RuntimeError('Configure both exact fixed Test manifest and bbox-cache paths, or leave both blank')
if not FIXED_TEST_MANIFEST_PATH:
    current_summary['fixed_opened_test2000']={'evaluated':False,'reason':'exact prior manifest/cache paths not supplied'}
    strict_json_write(comparison_dir/'transfer_summary.json',current_summary)
    progress('TEST','Skipped; no exact prior Test manifest/cache paths configured')
else:
    manifest_path=Path(FIXED_TEST_MANIFEST_PATH).expanduser()
    cache_path=Path(FIXED_TEST_BBOX_CACHE_PATH).expanduser()
    if not manifest_path.is_file() or not cache_path.is_file():raise FileNotFoundError('Configured fixed Test manifest/cache missing')
    manifest=pd.read_csv(manifest_path)
    needed={'sample_key','three_class_label','attack_code'}
    if not needed.issubset(manifest.columns) or len(manifest)!=2000 or manifest['sample_key'].duplicated().any():
        raise RuntimeError('Fixed opened Test-2000 manifest must contain exactly 2,000 unique keys and labels')
    test_keys=manifest['sample_key'].astype(str).tolist()
    if not all(k.startswith('Data/test/') for k in test_keys):raise RuntimeError('Manifest includes non-Test keys')
    if ordered_path_fingerprint(test_keys)!=PREVIOUS_OPENED_TEST_ORDERED_SHA256:
        raise RuntimeError('Manifest is not the exact previously opened Test-2000 identity/order')
    if set(test_keys)&(set(resources['train_keys'])|set(resources['val_keys'])):raise RuntimeError('Test keys overlap frozen Train/Val')
    test_labels=manifest['three_class_label'].astype(int).to_numpy()
    test_codes=manifest['attack_code'].astype(int).to_numpy()
    if not np.array_equal(test_labels,labels_from_attack_codes(test_codes)):raise RuntimeError('Test labels/codes disagree')
    cache=strict_json_load(cache_path);records=cache.get('records',{})
    if set(records)!=set(test_keys) or cache.get('ordered_path_fingerprint_sha256')!=ordered_path_fingerprint(test_keys):
        raise RuntimeError('Test bbox records do not match exact fixed manifest')
    for key in test_keys:validate_bbox_record(records[key],'celeba',key)
    missing=[k for k in test_keys if not (celeba_root/k).is_file()]
    if missing:raise FileNotFoundError(f'{len(missing)} selected Test images missing; first={missing[:3]}')
    test_ds=CelebAMicroDataset(test_keys,test_labels,test_codes,celeba_root,records,resources['stage0_dir'],train=False)
    test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,drop_last=False,
        num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=NUM_WORKERS>0)
    test_rows=[]
    for result,filename in ((vanilla,'vanilla_test_predictions.csv'),(csmr,'csmr_test_predictions.csv')):
        if sha256_file(result['out_dir']/'best.pth')!=result['checkpoint_sha256']:raise RuntimeError('Selected checkpoint SHA changed')
        state=torch.load(result['out_dir']/'best.pth',map_location='cpu',weights_only=False)
        result['model'].load_state_dict(state['state_dict'],strict=True);result['model'].to(device)
        logits,y,keys,codes=eval_spatial(result['model'],test_loader,device,use_amp)
        assert_ordered_keys(keys,test_keys,result['name']+' fixed opened Test')
        if not np.array_equal(y,test_labels) or not np.array_equal(codes,test_codes):raise RuntimeError('Test labels/order changed')
        metrics,_=metrics_celeba(y,logits,result['locked_threshold'])
        if metrics['n_scored']!=2000:raise RuntimeError('Test coverage incomplete; no resampling')
        make_prediction_frame(keys,y,codes,logits,result['locked_threshold']).to_csv(comparison_dir/filename,index=False)
        result['test']=metrics;result['model'].cpu();torch.cuda.empty_cache()
        progress(result['name'],f'Fixed opened Test ACER={metrics["acer"]:.6f} AUC={metrics["auc"]:.6f}')
    for metric in ('apcer','bpcer','acer','auc'):
        test_rows.append({'metric':'Test '+metric.upper(),'vanilla':vanilla['test'][metric],
            'csmr':csmr['test'][metric],'delta_csmr_minus_vanilla':csmr['test'][metric]-vanilla['test'][metric]})
    pd.DataFrame(test_rows).to_csv(comparison_dir/'test2000_metrics.csv',index=False)
    current_summary['fixed_opened_test2000']={'evaluated':True,'role':'fixed opened descriptive benchmark',
        'manifest_sha256':sha256_file(manifest_path),'bbox_cache_sha256':sha256_file(cache_path),
        'ordered_path_fingerprint_sha256':ordered_path_fingerprint(test_keys),
        'n':2000,'vanilla':vanilla['test'],'csmr':csmr['test'],
        'delta_csmr_minus_vanilla':{m:csmr['test'][m]-vanilla['test'][m] for m in ('apcer','bpcer','acer','auc')}}
    strict_json_write(comparison_dir/'transfer_summary.json',current_summary)
    progress('TEST','Saved fixed opened Test predictions and metrics; development verdict unchanged')


In [ ]:
report_path=comparison_dir/'comparison_report.md'
report_lines=report_path.read_text().splitlines()
if current_summary.get('fixed_opened_test2000',{}).get('evaluated'):
    r=current_summary['fixed_opened_test2000']
    report_lines += ['', '## FIXED OPENED TEST-2000 DESCRIPTIVE BENCHMARK',
        f'Exact manifest SHA `{r["manifest_sha256"]}` and bbox-cache SHA `{r["bbox_cache_sha256"]}`; no resampling. Each checkpoint used its own locked micro-Val threshold. Test did not select checkpoints, thresholds, or recipe.','',
        '| Metric | Vanilla | CSMR | Delta |','|---|---:|---:|---:|']
    for row in test_rows:report_lines.append(f'| {row["metric"]} | {row["vanilla"]:.6f} | {row["csmr"]:.6f} | {row["delta_csmr_minus_vanilla"]:+.6f} |')
else:
    report_lines += ['', '## FIXED OPENED TEST-2000 DESCRIPTIVE BENCHMARK',
        'Skipped because exact prior manifest and bbox-cache paths were not supplied.']
report_path.write_text('\n'.join(report_lines)+'\n')
progress('REPORT',f'Final comparison report={report_path}; summary={comparison_dir/"transfer_summary.json"}')


In [ ]:
# LAST CELL: ZIP only this run's outputs, never input datasets or crop cache.
zip_path=Path('/kaggle/working/mnv3s_crop27_csmr_seed100.zip')
if zip_path.exists():raise RuntimeError(f'ZIP collision: {zip_path}; start a clean Kaggle session')
write_zip(output_root,zip_path)
progress('ZIP',f'{zip_path} | {zip_path.stat().st_size/2**20:.2f} MB')
display(FileLink(str(zip_path)))
